# Training Notebooks

In [1]:
var="/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
    "$var"/tifffile-2025.10.16-py3-none-any.whl \
    "$var"/imagecodecs-2025.11.11-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl \
    "$var"/medicai-0.0.3-py3-none-any.whl \
    --no-index \
    --find-links "$var"

Looking in links: /kaggle/input/vsdetection-packages-offline-installer-only/whls
Processing /kaggle/input/vsdetection-packages-offline-installer-only/whls/tifffile-2025.10.16-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/vsdetection-packages-offline-installer-only/whls/tifffile-2025.10.16-py3-none-any.whl'



In [2]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
from medicai.transforms import (
    Compose,
    ScaleIntensityRange,
)
import torch
from medicai.models import SegFormer, TransUNet
from medicai.utils.inference import SlidingWindowInference
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import zipfile
import tensorflow as tf
import tifffile
from tqdm import tqdm
from keras import ops
from matplotlib import pyplot as plt

keras.config.backend(), keras.version()

ModuleNotFoundError: No module named 'medicai'

**Transformation**

In [ ]:
import tensorflow as tf
import numpy as np
from medicai.transforms import (
    Compose,
    ScaleIntensityRange,
    RandShiftIntensity,
    RandRotate90,
    RandFlip,
    RandSpatialCrop
)

# ==================== FIXED CUSTOM TRANSFORMS ====================

class RandAdjustContrast:
    """Random Contrast Adjustment (Gamma Correction) - TF Graph Compatible"""
    def __init__(self, keys, prob=0.3, gamma_range=(0.7, 1.3)):
        self.keys = keys
        self.prob = prob
        self.gamma_range = gamma_range
    
    def __call__(self, data):
        for key in self.keys:
            # Use tf.cond instead of Python if
            data[key] = tf.cond(
                tf.random.uniform([]) < self.prob,
                lambda: self._apply_gamma(data[key]),
                lambda: data[key]
            )
        return data
    
    def _apply_gamma(self, image):
        gamma = tf.random.uniform(
            [], 
            self.gamma_range[0], 
            self.gamma_range[1]
        )
        # Clip to [0, 1] before applying gamma
        image_clipped = tf.clip_by_value(image, 0.0, 1.0)
        return tf.pow(image_clipped, gamma)


class RandUnsharpMask:
    """Random Unsharp Masking - TF Graph Compatible"""
    def __init__(self, keys, prob=0.3, amount_range=(0.5, 1.5), kernel_size=3):
        self.keys = keys
        self.prob = prob
        self.amount_range = amount_range
        self.kernel_size = kernel_size
    
    def __call__(self, data):
        for key in self.keys:
            data[key] = tf.cond(
                tf.random.uniform([]) < self.prob,
                lambda: self._apply_unsharp_mask(data[key]),
                lambda: data[key]
            )
        return data
    
    def _apply_unsharp_mask(self, image):
        amount = tf.random.uniform(
            [], 
            self.amount_range[0], 
            self.amount_range[1]
        )
        
        # Simple box blur untuk 3D
        blurred = self._box_blur_3d(image, self.kernel_size)
        
        # Unsharp mask formula
        sharpened = image + amount * (image - blurred)
        return tf.clip_by_value(sharpened, 0.0, 1.0)
    
    def _box_blur_3d(self, volume, kernel_size):
        """Box blur menggunakan average pooling 3D"""
        # Add batch dimension
        volume_batched = volume[tf.newaxis, ...]
        
        # Apply 3D average pooling
        ksize = [1, kernel_size, kernel_size, kernel_size, 1]
        strides = [1, 1, 1, 1, 1]
        blurred = tf.nn.avg_pool3d(
            volume_batched,
            ksize=ksize,
            strides=strides,
            padding='SAME'
        )
        
        # Remove batch dimension
        return blurred[0]


class RandAddGaussianNoise:
    """Random Gaussian Noise - TF Graph Compatible"""
    def __init__(self, keys, prob=0.2, noise_std_range=(0.0, 0.05)):
        self.keys = keys
        self.prob = prob
        self.noise_std_range = noise_std_range
    
    def __call__(self, data):
        for key in self.keys:
            data[key] = tf.cond(
                tf.random.uniform([]) < self.prob,
                lambda: self._add_noise(data[key]),
                lambda: data[key]
            )
        return data
    
    def _add_noise(self, image):
        noise_std = tf.random.uniform(
            [], 
            self.noise_std_range[0], 
            self.noise_std_range[1]
        )
        
        noise = tf.random.normal(
            tf.shape(image),
            mean=0.0,
            stddev=noise_std
        )
        
        noisy = image + noise
        return tf.clip_by_value(noisy, 0.0, 1.0)


class RandBrightnessContrast:
    """Random Brightness and Contrast adjustment - TF Graph Compatible"""
    def __init__(self, keys, prob=0.3, 
                 brightness_range=(-0.1, 0.1),
                 contrast_range=(0.8, 1.2)):
        self.keys = keys
        self.prob = prob
        self.brightness_range = brightness_range
        self.contrast_range = contrast_range
    
    def __call__(self, data):
        for key in self.keys:
            data[key] = tf.cond(
                tf.random.uniform([]) < self.prob,
                lambda: self._adjust_brightness_contrast(data[key]),
                lambda: data[key]
            )
        return data
    
    def _adjust_brightness_contrast(self, image):
        # Random brightness adjustment
        brightness_delta = tf.random.uniform(
            [],
            self.brightness_range[0],
            self.brightness_range[1]
        )
        image = image + brightness_delta
        
        # Random contrast adjustment
        contrast_factor = tf.random.uniform(
            [],
            self.contrast_range[0],
            self.contrast_range[1]
        )
        mean = tf.reduce_mean(image)
        image = (image - mean) * contrast_factor + mean
        
        return tf.clip_by_value(image, 0.0, 1.0)


class RandHistogramShift:
    """Random Histogram Shifting - Simplified TF Graph Compatible"""
    def __init__(self, keys, prob=0.3, shift_range=(-0.1, 0.1)):
        self.keys = keys
        self.prob = prob
        self.shift_range = shift_range
    
    def __call__(self, data):
        for key in self.keys:
            data[key] = tf.cond(
                tf.random.uniform([]) < self.prob,
                lambda: self._shift_histogram(data[key]),
                lambda: data[key]
            )
        return data
    
    def _shift_histogram(self, image):
        """Simplified histogram shift using intensity scaling"""
        shift = tf.random.uniform(
            [],
            self.shift_range[0],
            self.shift_range[1]
        )
        
        # Non-linear transformation
        shifted = image + shift * (image - 0.5)
        return tf.clip_by_value(shifted, 0.0, 1.0)


# ==================== PIPELINE DENGAN FIXED TRANSFORMS ====================

def train_transformation(image, label):
    """Training transformation dengan custom enhancement - FIXED VERSION"""
    data = {"image": image, "label": label}
    
    pipeline = Compose([
        # 1. Normalisasi dasar
        ScaleIntensityRange(
            keys=["image"],
            a_min=0,
            a_max=255,
            clip=True,
        ),
        
        # 2. Spatial augmentation (dari medicai)
        RandSpatialCrop(
            keys=["image", "label"],
            roi_size=(128, 128, 128),
        ),
        RandFlip(keys=["image", "label"], spatial_axis=[0], prob=0.5),
        RandFlip(keys=["image", "label"], spatial_axis=[1], prob=0.5),
        RandFlip(keys=["image", "label"], spatial_axis=[2], prob=0.5),
        RandRotate90(
            keys=["image", "label"], 
            prob=0.4, 
            max_k=3, 
            spatial_axes=(0, 1)
        ),
        
        # 3. CUSTOM ENHANCEMENT TRANSFORMS (FIXED)
        RandAdjustContrast(
            keys=["image"],
            prob=0.3,
            gamma_range=(0.7, 1.3)
        ),
        
        RandUnsharpMask(
            keys=["image"],
            prob=0.25,
            amount_range=(0.5, 1.2),
            kernel_size=3
        ),
        
        RandBrightnessContrast(
            keys=["image"],
            prob=0.3,
            brightness_range=(-0.1, 0.1),
            contrast_range=(0.85, 1.15)
        ),
        
        RandAddGaussianNoise(
            keys=["image"],
            prob=0.2,
            noise_std_range=(0.0, 0.03)
        ),
        
        RandHistogramShift(
            keys=["image"],
            prob=0.2,
            shift_range=(-0.05, 0.05)
        ),
        
        # 4. Intensity shift (dari medicai)
        RandShiftIntensity(
            keys=["image"], 
            offsets=0.10, 
            prob=0.5
        ),
    ])
    
    result = pipeline(data)
    return result["image"], result["label"]


def val_transformation(image, label):
    """Validation transform - hanya normalisasi"""
    data = {"image": image, "label": label}
    pipeline = Compose([
        ScaleIntensityRange(
            keys=["image"],
            a_min=0,
            a_max=255,
            clip=True,
        ),
    ])
    result = pipeline(data)
    return result["image"], result["label"]

**Model**

In [ ]:
import medicai
from medicai.losses import BinaryDiceCELoss
from medicai.metrics import BinaryDiceMetric

In [ ]:
num_classes = 1
model = SegFormer(
    input_shape=(128, 128, 128, 1),
    encoder_name='mit_b0',
    classifier_activation='sigmoid',
    num_classes=num_classes,
)
model.count_params() / 1e6

In [ ]:
# define optomizer, loss
optim = keras.optimizers.AdamW(
    learning_rate=1e-4,
    weight_decay=1e-5,
)
loss_fn = BinaryDiceCELoss(
    from_logits=False, 
    num_classes=num_classes
)

# define sliding-window-inferencer for validation
swi = SlidingWindowInference(
    model,
    num_classes=num_classes,
    roi_size=(128, 128, 128),
    sw_batch_size=4,
    overlap=0.5,
)

In [ ]:
def train_one_epoch(model, dataloader, metrics):
    loop = tqdm(dataloader, desc="Training", leave=False)
    
    for imgs, labels in loop:
        # forward pass
        outputs = model(imgs)
        loss = loss_fn(labels, outputs)

        # backward pass
        model.zero_grad()
        trainable_weights = [v for v in model.trainable_weights]

        # call torch.Tensor.backward() on the loss to compute gradients
        loss.backward()
        gradients = [v.value.grad for v in trainable_weights]

        # update weights
        with torch.no_grad():
            optim.apply(gradients, trainable_weights)

        # update training metric
        y_true = ops.convert_to_tensor(labels)
        y_pred = ops.convert_to_tensor(outputs)
        
        # Kalau 5D: (B, D, H, W, C) → flatten jadi (B, D*H*W, C)
        if len(y_true.shape) == 5:
            b, d, h, w, c = y_true.shape
            y_true = ops.reshape(y_true, (b, d * h * w, c))
            y_pred = ops.reshape(y_pred, (b, d * h * w, c))
        
        metrics.update_state(y_true, y_pred)

        
        # Update tqdm
        loss_score = ops.convert_to_numpy(loss)
        metrics_score = ops.convert_to_numpy(metrics.result())
        loop.set_postfix(
            loss=loss_score,
            dice=metrics_score,
        )

    return loss, metrics

In [ ]:
def validate(model, dataloader, metrics):
    # optional tapi bagus: set ke eval
    model.eval()

    for x, y in dataloader:
        # kalau pakai torch backend, sebaiknya no_grad biar hemat memori
        with torch.no_grad():
            output = swi(x)  # atau model(x), sesuai punyamu

        # konversi ke tensor keras.ops
        y = ops.convert_to_tensor(y)
        output = ops.convert_to_tensor(output)

        # EXPECTED SHAPE awal:
        # y      : (B, D, H, W, C)
        # output : (B, D, H, W, C)
        if len(y.shape) == 5:
            b, d, h, w, c = y.shape  # (batch, depth, height, width, channel)

            # flatten dim spasial → (B, N, C), N = D*H*W
            y = ops.reshape(y, (b, d * h * w, c))
            output = ops.reshape(output, (b, d * h * w, c))

        # urutan: (y_true, y_pred)
        metrics.update_state(y, output)

    return metrics


In [ ]:
def run_training(train_loader, val_loader, model, epochs=20):
    # metrics for train
    train_metrics = BinaryDiceMetric(
        from_logits=False, 
        num_classes=num_classes, 
        name='dice'
    )

    # metrics for validation
    val_metrics = BinaryDiceMetric(
        from_logits=False, 
        num_classes=num_classes, 
        name='val_dice'
    )
    
    # Initialize best validation dice score
    best_val_dice = 0.0

    for epoch in range(epochs):
        print(f'Epoch {epoch+1}/{epochs}')
        
        # Training
        loss, train_metrics = train_one_epoch(
            model, train_loader, train_metrics,
        )
        # display training logs at the end of epoch
        train_metrics_score = ops.convert_to_numpy(train_metrics.result())
        loss_score = ops.convert_to_numpy(loss)
        
        # reset training metrics at the end of each epoch
        train_metrics.reset_state()

        # Validation [at every 5 epoch]
        if (epoch + 1) % 5 == 0:
            val_metrics = validate(model, val_loader, val_metrics)
            val_metrics_score = ops.convert_to_numpy(
                val_metrics.result()
            )
            val_metrics.reset_state()
            print(
                f'Training - Loss: {loss_score:.4f}, Dice: {train_metrics_score:.4f}'
                f'\nValidation - Dice: {val_metrics_score:.4f}\n'
            )

            # Save best model weights
            if val_metrics_score > best_val_dice:
                best_val_dice = val_metrics_score
                model.save_weights('model.weights.h5')
                # torch.save(model.state_dict(), 'model.pth') # OK too.
                print(
                    f'Dice score improved: {best_val_dice}. Model saved.'
                )
        else:
            print(
                f'Training - Loss: {loss_score:.4f}, Dice: {train_metrics_score:.4f}\n'
            )

**Sliding Window Inference**

In [ ]:
import tensorflow as tf
import numpy as np
import tifffile

def _read_tiff(path):
    # path: tf.string → numpy bytes → decode
    path = path.decode("utf-8")
    vol = tifffile.imread(path).astype(np.uint8)  # (D, H, W) 3D volume
    return vol

def parse_tiff_fn(image_path, label_path):
    # Baca image & label dari path dengan numpy_function
    image = tf.numpy_function(_read_tiff, [image_path], tf.uint8)
    label = tf.numpy_function(_read_tiff, [label_path], tf.uint8)

    # Set shape ke 3D volume (D, H, W); dimensi exact-nya unknown (None)
    image.set_shape((None, None, None))
    label.set_shape((None, None, None))

    # Kembalikan PERSIS seperti input ke prepare_inputs: (image, label)
    return image, label


In [ ]:
def prepare_inputs(image, label):
    # Only take gt 1 
    label = (label == 1)
    
    # Add channel dimension
    image = image[..., None] # (D, H, W, 1)
    label = label[..., None] # (D, H, W, 1)

    # Convert to float32
    image = tf.cast(image, tf.float32)
    label = tf.cast(label, tf.float32)
    
    return image, label


In [ ]:
def load_tiff_dataset(dataset, batch_size=1, shuffle=True):
    # Ambil list path
    image_paths = [sample["image"] for sample in dataset]
    label_paths = [sample["label"] for sample in dataset]

    ds = tf.data.Dataset.from_tensor_slices((image_paths, label_paths))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(image_paths))

    # 1) Baca TIFF → (image, label) uint8, shape (D,H,W)
    ds = ds.map(
        parse_tiff_fn,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # 2) Apply prepare_inputs (sama persis kaya pipeline TFRecord)
    ds = ds.map(
        prepare_inputs,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # 3) Augment / transform sesuai mode
    if shuffle:
        ds = ds.map(
            train_transformation,
            num_parallel_calls=tf.data.AUTOTUNE
        )
    else:
        ds = ds.map(
            val_transformation,
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # 4) Batch + prefetch
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
import random

def split_dataset(dataset, val_ratio=0.2, seed=42):
    random.seed(seed)
    dataset = dataset.copy()
    random.shuffle(dataset)

    n_total = len(dataset)
    n_val = int(n_total * val_ratio)

    val_set = dataset[:n_val]
    train_set = dataset[n_val:]

    return train_set, val_set

In [ ]:
import glob
import os

image_dir = "/kaggle/input/vesuvius-challenge-surface-detection/train_images"
label_dir = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels"

image_paths = sorted(glob.glob(os.path.join(image_dir, "*.tif")))
label_paths = sorted(glob.glob(os.path.join(label_dir, "*.tif")))

dataset = [
    {"image": im_path, "label": lb_path}
    for im_path, lb_path in zip(image_paths, label_paths)
]

train_set, val_set = split_dataset(dataset)

train_ds = load_tiff_dataset(train_set, batch_size=6, shuffle=True)
val_ds = load_tiff_dataset(train_set, batch_size=1, shuffle=False)


In [ ]:
# x, y = next(iter(train_ds))
# x.shape, y.shape

In [ ]:
run_training(
    train_ds, val_ds, model, epochs=20
)

In [ ]:
import tifffile as tiff
import numpy as np

def load_tiff_3d_np(path):
    volume = tiff.imread(path)      # bisa return (D, H, W) atau (H, W, D) atau (H, W)
    volume = volume.astype("float32")

    # Normalisasi 0–1, asumsi 16-bit TIFF (Vesuvius biasanya 0–65535)
    max_val = volume.max() if volume.max() > 0 else 1.0
    volume = volume / max_val

    # Pastikan ada channel dimension
    if volume.ndim == 3:
        # asumsikan (D, H, W) → tambahkan channel=1 → (D, H, W, 1)
        volume = volume[..., np.newaxis]
    elif volume.ndim == 2:
        # (H, W) → (1, H, W, 1) misal dianggap depth=1
        volume = volume[np.newaxis, ..., np.newaxis]

    return volume

path = "/kaggle/input/vesuvius-challenge-surface-detection/test_images/1407735.tif"
vol_np = load_tiff_3d_np(path)
print("numpy shape:", vol_np.shape)

In [ ]:
res = swi(np.expand_dims(vol_np, axis=0))
segres = (res > 0.35).astype(np.uint8)
tiff.imwrite("/kaggle/working/1407735.tif", segres[0])

In [ ]:
import zipfile
import os

output_zip = "submission.zip"
pred_dir = "/kaggle/working"

with zipfile.ZipFile(output_zip, "w") as zipf:
    for filename in os.listdir(pred_dir):
        if filename.endswith(".tif"):
            zipf.write(
                os.path.join(pred_dir, filename),
                arcname=filename  # penting: hanya nama file, tanpa path
            )

print("ZIP created:", output_zip)